In [3]:
# 0. 프로젝트 설명 메모

"""
Project: Olist 이커머스 주문·배송·리뷰 데이터 기반 고객 경험 분석
Folder: olist-ecommerce-analysis

1단계 목적
- Kaggle Olist Brazilian E-Commerce Public Dataset CSV 9개를 로드한다.
- 각 테이블의 구조, 컬럼, 결측치, 키 중복 여부를 확인한다.
- SQLite DB에 적재하여 SQL 기반 분석 환경을 만든다.
- 주요 테이블 관계가 정상적으로 JOIN되는지 검증한다.

주의
- 이 데이터는 클릭, 장바구니, 페이지뷰 같은 웹 행동 로그가 아니다.
- 주문, 결제, 배송, 리뷰 중심의 거래 데이터다.
- 따라서 본 프로젝트는 퍼널/코호트 분석이 아니라 주문·배송·리뷰 기반 고객 경험 분석으로 해석한다.
"""


## 1. 라이브러리 import
import os
import sqlite3
import pandas as pd
import numpy as np
from pathlib import Path


## 2. 경로 설정
# 실제 프로젝트 경로 기준으로 고정
PROJECT_DIR = Path(r"C:\취준\olist-ecommerce-analysis")

DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "outputs"
DB_DIR = PROJECT_DIR / "db"

OUTPUT_DIR.mkdir(exist_ok=True)
DB_DIR.mkdir(exist_ok=True)

DB_PATH = DB_DIR / "olist_ecommerce.db"

print("PROJECT_DIR:", PROJECT_DIR)
print("DATA_DIR:", DATA_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("DB_DIR:", DB_DIR)
print("DB_PATH:", DB_PATH)


## 3. CSV 파일 목록 정의
csv_files = {
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "products": "olist_products_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}

PROJECT_DIR: C:\취준\olist-ecommerce-analysis
DATA_DIR: C:\취준\olist-ecommerce-analysis\data
OUTPUT_DIR: C:\취준\olist-ecommerce-analysis\outputs
DB_DIR: C:\취준\olist-ecommerce-analysis\db
DB_PATH: C:\취준\olist-ecommerce-analysis\db\olist_ecommerce.db


In [4]:
## 4. CSV 9개 존재 여부 확인
missing_files = []

for table_name, file_name in csv_files.items():
    file_path = DATA_DIR / file_name
    
    if file_path.exists():
        print(f"[OK] {table_name:22s} -> {file_name}")
    else:
        print(f"[MISSING] {table_name:22s} -> {file_name}")
        missing_files.append(file_name)

if missing_files:
    raise FileNotFoundError(f"누락된 파일이 있습니다: {missing_files}")
else:
    print("\nCSV 9개 파일이 모두 정상적으로 확인되었습니다.")

## 5. CSV 로드
dfs = {}

for table_name, file_name in csv_files.items():
    file_path = DATA_DIR / file_name
    dfs[table_name] = pd.read_csv(file_path)
    print(f"{table_name:22s} loaded: {dfs[table_name].shape}")

[OK] orders                 -> olist_orders_dataset.csv
[OK] order_items            -> olist_order_items_dataset.csv
[OK] order_payments         -> olist_order_payments_dataset.csv
[OK] order_reviews          -> olist_order_reviews_dataset.csv
[OK] customers              -> olist_customers_dataset.csv
[OK] sellers                -> olist_sellers_dataset.csv
[OK] products               -> olist_products_dataset.csv
[OK] geolocation            -> olist_geolocation_dataset.csv
[OK] category_translation   -> product_category_name_translation.csv

CSV 9개 파일이 모두 정상적으로 확인되었습니다.
orders                 loaded: (99441, 8)
order_items            loaded: (112650, 7)
order_payments         loaded: (103886, 5)
order_reviews          loaded: (99224, 7)
customers              loaded: (99441, 5)
sellers                loaded: (3095, 4)
products               loaded: (32951, 9)
geolocation            loaded: (1000163, 5)
category_translation   loaded: (71, 2)


In [5]:
## 6. 테이블별 기본 구조 확인
table_summary = []

for table_name, df in dfs.items():
    table_summary.append({
        "table_name": table_name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "duplicated_rows": df.duplicated().sum(),
        "total_missing_values": df.isna().sum().sum()
    })

table_summary_df = pd.DataFrame(table_summary)
table_summary_df

table_summary_df.to_csv(
    OUTPUT_DIR / "01_table_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

## 7. 테이블별 컬럼 확인
for table_name, df in dfs.items():
    print("=" * 80)
    print(f"[{table_name}]")
    print(f"shape: {df.shape}")
    print(df.columns.tolist())

## 8. 테이블별 데이터 타입 확인
for table_name, df in dfs.items():
    print("=" * 80)
    print(f"[{table_name}] dtypes")
    print(df.dtypes)

## 9. 결측치 확인
missing_summary_list = []

for table_name, df in dfs.items():
    missing_count = df.isna().sum()
    missing_rate = df.isna().mean() * 100
    
    temp = pd.DataFrame({
        "table_name": table_name,
        "column_name": df.columns,
        "missing_count": missing_count.values,
        "missing_rate_pct": missing_rate.values
    })
    
    temp = temp[temp["missing_count"] > 0]
    missing_summary_list.append(temp)

missing_summary_df = pd.concat(missing_summary_list, ignore_index=True)
missing_summary_df = missing_summary_df.sort_values(
    ["table_name", "missing_rate_pct"],
    ascending=[True, False]
)

missing_summary_df

missing_summary_df.to_csv(
    OUTPUT_DIR / "02_missing_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

## 10. 주요 키 중복 여부 확인
key_checks = [
    ("customers", "customer_id"),
    ("customers", "customer_unique_id"),
    ("orders", "order_id"),
    ("order_items", "order_id"),
    ("order_items", "product_id"),
    ("order_items", "seller_id"),
    ("order_payments", "order_id"),
    ("order_reviews", "order_id"),
    ("products", "product_id"),
    ("sellers", "seller_id"),
    ("geolocation", "geolocation_zip_code_prefix"),
    ("category_translation", "product_category_name"),
]

key_summary = []

for table_name, key_col in key_checks:
    df = dfs[table_name]
    
    key_summary.append({
        "table_name": table_name,
        "key_column": key_col,
        "rows": len(df),
        "non_null_count": df[key_col].notna().sum(),
        "unique_count": df[key_col].nunique(dropna=True),
        "duplicated_count": df[key_col].duplicated().sum()
    })

key_summary_df = pd.DataFrame(key_summary)
key_summary_df

key_summary_df.to_csv(
    OUTPUT_DIR / "03_key_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

[orders]
shape: (99441, 8)
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']
[order_items]
shape: (112650, 7)
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']
[order_payments]
shape: (103886, 5)
['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']
[order_reviews]
shape: (99224, 7)
['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']
[customers]
shape: (99441, 5)
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']
[sellers]
shape: (3095, 4)
['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']
[products]
shape: (32951, 9)
['product_id', 'product_category_name', 'product_name_lenght', 'product_descript

In [6]:
## 11. 주문 상태 분포 확인
orders = dfs["orders"]

order_status_summary = (
    orders["order_status"]
    .value_counts()
    .reset_index()
)

order_status_summary.columns = ["order_status", "order_count"]
order_status_summary["order_rate_pct"] = (
    order_status_summary["order_count"] 
    / order_status_summary["order_count"].sum() 
    * 100
).round(2)

order_status_summary

order_status_summary.to_csv(
    OUTPUT_DIR / "04_order_status_summary.csv",
    index=False,
    encoding="utf-8-sig"
)


## 12. 날짜 컬럼 변환
date_columns = {
    "orders": [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ],
    "order_items": [
        "shipping_limit_date"
    ],
    "order_reviews": [
        "review_creation_date",
        "review_answer_timestamp"
    ]
}

for table_name, cols in date_columns.items():
    for col in cols:
        dfs[table_name][col] = pd.to_datetime(dfs[table_name][col], errors="coerce")

print("날짜 컬럼 변환 완료")


## 13. 주문 기간 확인
orders = dfs["orders"]

print("전체 주문일 최소:", orders["order_purchase_timestamp"].min())
print("전체 주문일 최대:", orders["order_purchase_timestamp"].max())

delivered_orders = orders[orders["order_status"] == "delivered"].copy()

print("delivered 주문 수:", len(delivered_orders))
print("delivered 주문일 최소:", delivered_orders["order_purchase_timestamp"].min())
print("delivered 주문일 최대:", delivered_orders["order_purchase_timestamp"].max())


## 14. 배송 관련 파생변수 생성
orders_enriched = dfs["orders"].copy()

orders_enriched["purchase_date"] = orders_enriched["order_purchase_timestamp"].dt.date
orders_enriched["purchase_year"] = orders_enriched["order_purchase_timestamp"].dt.year
orders_enriched["purchase_month"] = orders_enriched["order_purchase_timestamp"].dt.to_period("M").astype(str)
orders_enriched["purchase_dayofweek"] = orders_enriched["order_purchase_timestamp"].dt.day_name()

# 배송 소요일: 구매 시점부터 고객에게 실제 배송 완료까지 걸린 일수
orders_enriched["delivery_days"] = (
    orders_enriched["order_delivered_customer_date"] 
    - orders_enriched["order_purchase_timestamp"]
).dt.total_seconds() / (60 * 60 * 24)

# 예상 배송일 대비 실제 배송 지연일
# 양수면 예상일보다 늦게 도착, 0 이하이면 정시 또는 조기 도착
orders_enriched["delay_days"] = (
    orders_enriched["order_delivered_customer_date"] 
    - orders_enriched["order_estimated_delivery_date"]
).dt.total_seconds() / (60 * 60 * 24)

orders_enriched["is_delayed"] = np.where(orders_enriched["delay_days"] > 0, 1, 0)

orders_enriched[
    [
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "purchase_month",
        "delivery_days",
        "delay_days",
        "is_delayed"
    ]
].head()


## 15. Delivered 주문 기준 배송 지표 확인
delivered_enriched = orders_enriched[orders_enriched["order_status"] == "delivered"].copy()

delivery_basic_summary = pd.DataFrame({
    "metric": [
        "delivered_order_count",
        "avg_delivery_days",
        "median_delivery_days",
        "avg_delay_days",
        "median_delay_days",
        "delayed_order_count",
        "delayed_order_rate_pct"
    ],
    "value": [
        len(delivered_enriched),
        delivered_enriched["delivery_days"].mean(),
        delivered_enriched["delivery_days"].median(),
        delivered_enriched["delay_days"].mean(),
        delivered_enriched["delay_days"].median(),
        delivered_enriched["is_delayed"].sum(),
        delivered_enriched["is_delayed"].mean() * 100
    ]
})

delivery_basic_summary

delivered_enriched = orders_enriched[orders_enriched["order_status"] == "delivered"].copy()

delivery_basic_summary = pd.DataFrame({
    "metric": [
        "delivered_order_count",
        "avg_delivery_days",
        "median_delivery_days",
        "avg_delay_days",
        "median_delay_days",
        "delayed_order_count",
        "delayed_order_rate_pct"
    ],
    "value": [
        len(delivered_enriched),
        delivered_enriched["delivery_days"].mean(),
        delivered_enriched["delivery_days"].median(),
        delivered_enriched["delay_days"].mean(),
        delivered_enriched["delay_days"].median(),
        delivered_enriched["is_delayed"].sum(),
        delivered_enriched["is_delayed"].mean() * 100
    ]
})

delivery_basic_summary

delivery_basic_summary.to_csv(
    OUTPUT_DIR / "05_delivery_basic_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

날짜 컬럼 변환 완료
전체 주문일 최소: 2016-09-04 21:15:19
전체 주문일 최대: 2018-10-17 17:30:18
delivered 주문 수: 96478
delivered 주문일 최소: 2016-09-15 12:16:38
delivered 주문일 최대: 2018-08-29 15:00:37


In [7]:
## 16. SQLite DB 생성 및 적재
if DB_PATH.exists():
    DB_PATH.unlink()
    print("기존 DB 파일 삭제:", DB_PATH)

conn = sqlite3.connect(DB_PATH)

for table_name, df in dfs.items():
    df.to_sql(table_name, conn, if_exists="replace", index=False)
    print(f"[SAVED] {table_name:22s} -> SQLite")

orders_enriched.to_sql("orders_enriched", conn, if_exists="replace", index=False)
print("[SAVED] orders_enriched       -> SQLite")


## 17. SQLite 테이블 목록 확인
query = """
SELECT name
FROM sqlite_master
WHERE type = 'table'
ORDER BY name;
"""

pd.read_sql_query(query, conn)


## 18. SQLite 테이블별 row 수 확인
table_count_query = """
SELECT 'customers' AS table_name, COUNT(*) AS row_count FROM customers
UNION ALL
SELECT 'orders', COUNT(*) FROM orders
UNION ALL
SELECT 'orders_enriched', COUNT(*) FROM orders_enriched
UNION ALL
SELECT 'order_items', COUNT(*) FROM order_items
UNION ALL
SELECT 'order_payments', COUNT(*) FROM order_payments
UNION ALL
SELECT 'order_reviews', COUNT(*) FROM order_reviews
UNION ALL
SELECT 'products', COUNT(*) FROM products
UNION ALL
SELECT 'sellers', COUNT(*) FROM sellers
UNION ALL
SELECT 'geolocation', COUNT(*) FROM geolocation
UNION ALL
SELECT 'category_translation', COUNT(*) FROM category_translation;
"""

sqlite_table_counts = pd.read_sql_query(table_count_query, conn)
sqlite_table_counts

sqlite_table_counts.to_csv(
    OUTPUT_DIR / "06_sqlite_table_counts.csv",
    index=False,
    encoding="utf-8-sig"
)


## 19. SQLite 스키마 확인
tables_to_check = [
    "customers",
    "orders",
    "orders_enriched",
    "order_items",
    "order_payments",
    "order_reviews",
    "products",
    "sellers",
    "geolocation",
    "category_translation"
]

for table_name in tables_to_check:
    print("=" * 80)
    print(f"[{table_name}]")
    schema_df = pd.read_sql_query(f"PRAGMA table_info({table_name});", conn)
    display(schema_df)

[SAVED] orders                 -> SQLite
[SAVED] order_items            -> SQLite
[SAVED] order_payments         -> SQLite
[SAVED] order_reviews          -> SQLite
[SAVED] customers              -> SQLite
[SAVED] sellers                -> SQLite
[SAVED] products               -> SQLite
[SAVED] geolocation            -> SQLite
[SAVED] category_translation   -> SQLite
[SAVED] orders_enriched       -> SQLite
[customers]


,cid,name,type,notnull,dflt_value,pk
0,0,customer_id,TEXT,0,None,0
1,1,customer_unique_id,TEXT,0,None,0
2,2,customer_zip_code_prefix,INTEGER,0,None,0
3,3,customer_city,TEXT,0,None,0
4,4,customer_state,TEXT,0,None,0


[orders]


,cid,name,type,notnull,dflt_value,pk
0,0,order_id,TEXT,0,None,0
1,1,customer_id,TEXT,0,None,0
2,2,order_status,TEXT,0,None,0
3,3,order_purchase_timestamp,TIMESTAMP,0,None,0
4,4,order_approved_at,TIMESTAMP,0,None,0
5,5,order_delivered_carrier_date,TIMESTAMP,0,None,0
6,6,order_delivered_customer_date,TIMESTAMP,0,None,0
7,7,order_estimated_delivery_date,TIMESTAMP,0,None,0


[orders_enriched]


...

,cid,name,type,notnull,dflt_value,pk
0,0,seller_id,TEXT,0,None,0
1,1,seller_zip_code_prefix,INTEGER,0,None,0
2,2,seller_city,TEXT,0,None,0
3,3,seller_state,TEXT,0,None,0


[geolocation]


,cid,name,type,notnull,dflt_value,pk
0,0,geolocation_zip_code_prefix,INTEGER,0,None,0
1,1,geolocation_lat,REAL,0,None,0
2,2,geolocation_lng,REAL,0,None,0
3,3,geolocation_city,TEXT,0,None,0
4,4,geolocation_state,TEXT,0,None,0


[category_translation]


,cid,name,type,notnull,dflt_value,pk
0,0,product_category_name,TEXT,0,None,0
1,1,product_category_name_english,TEXT,0,None,0


In [8]:
## 20. 핵심 JOIN 검증 1 — customers → orders
query = """
/*
목적:
customers 테이블과 orders 테이블이 customer_id 기준으로 정상 연결되는지 확인한다.

해석:
orders.customer_id는 주문 단위 고객 ID이며,
customers.customer_id와 연결된다.
*/

SELECT
    COUNT(*) AS joined_rows,
    COUNT(DISTINCT o.order_id) AS distinct_orders,
    COUNT(DISTINCT c.customer_id) AS distinct_customer_ids,
    COUNT(DISTINCT c.customer_unique_id) AS distinct_unique_customers
FROM orders o
LEFT JOIN customers c
    ON o.customer_id = c.customer_id;
"""

join_check_customers_orders = pd.read_sql_query(query, conn)
join_check_customers_orders


## 21. 핵심 JOIN 검증 2 — orders → order_items → products
query = """
/*
목적:
주문, 주문상품, 상품 테이블이 정상적으로 연결되는지 확인한다.

주의:
order_items는 주문 1건에 여러 상품 row가 존재할 수 있으므로,
JOIN 결과 row 수는 orders 수보다 많아지는 것이 정상이다.
*/

SELECT
    COUNT(*) AS joined_rows,
    COUNT(DISTINCT o.order_id) AS distinct_orders,
    COUNT(DISTINCT oi.product_id) AS distinct_products_in_items,
    COUNT(DISTINCT p.product_id) AS distinct_products_joined
FROM orders o
LEFT JOIN order_items oi
    ON o.order_id = oi.order_id
LEFT JOIN products p
    ON oi.product_id = p.product_id;
"""

join_check_orders_items_products = pd.read_sql_query(query, conn)
join_check_orders_items_products


## 22. 핵심 JOIN 검증 3 — products → category_translation
query = """
/*
목적:
포르투갈어 상품 카테고리명을 영어 카테고리명으로 변환할 수 있는지 확인한다.

주의:
일부 상품은 product_category_name이 결측일 수 있다.
일부 카테고리는 translation 테이블에 없을 수 있다.
*/

SELECT
    COUNT(*) AS product_rows,
    COUNT(p.product_category_name) AS non_null_category_count,
    COUNT(t.product_category_name_english) AS translated_category_count,
    COUNT(*) - COUNT(t.product_category_name_english) AS not_translated_or_null_count
FROM products p
LEFT JOIN category_translation t
    ON p.product_category_name = t.product_category_name;
"""

join_check_products_translation = pd.read_sql_query(query, conn)
join_check_products_translation


## 23. 핵심 JOIN 검증 4 — orders → payments
query = """
/*
목적:
주문과 결제 데이터가 정상적으로 연결되는지 확인한다.

주의:
order_payments는 한 주문에 여러 결제 row가 존재할 수 있다.
예: 쿠폰, 카드 분할, 여러 결제 수단 등
따라서 주문별 결제금액 분석 시 먼저 order_id 기준으로 합산해야 한다.
*/

SELECT
    COUNT(*) AS joined_rows,
    COUNT(DISTINCT o.order_id) AS distinct_orders,
    COUNT(DISTINCT p.order_id) AS distinct_orders_with_payment,
    SUM(p.payment_value) AS total_payment_value
FROM orders o
LEFT JOIN order_payments p
    ON o.order_id = p.order_id;
"""

join_check_orders_payments = pd.read_sql_query(query, conn)
join_check_orders_payments


## 24. 핵심 JOIN 검증 5 — orders → reviews
query = """
/*
목적:
주문과 리뷰 데이터가 정상적으로 연결되는지 확인한다.

주의:
대부분 주문에는 리뷰가 있지만,
일부 주문은 리뷰가 없거나 한 주문에 리뷰 row가 2개 이상 있을 수 있다.
리뷰 분석 시 order_id 기준 중복 여부를 확인해야 한다.
*/

SELECT
    COUNT(*) AS joined_rows,
    COUNT(DISTINCT o.order_id) AS distinct_orders,
    COUNT(DISTINCT r.order_id) AS distinct_orders_with_review,
    AVG(r.review_score) AS avg_review_score
FROM orders o
LEFT JOIN order_reviews r
    ON o.order_id = r.order_id;
"""

join_check_orders_reviews = pd.read_sql_query(query, conn)
join_check_orders_reviews


## 25. delivered 주문 중심 통합 JOIN 샘플 생성
query = """
/*
목적:
delivered 주문을 기준으로 고객, 주문상품, 상품 카테고리, 판매자, 리뷰를 결합한 샘플을 확인한다.

분석 관점:
- 주문/배송 정보: orders_enriched
- 고객 지역: customers
- 상품/가격: order_items, products
- 카테고리 영어명: category_translation
- 판매자 지역: sellers
- 리뷰 점수: order_reviews

주의:
주문 1건에 상품이 여러 개 있으면 order_id가 여러 행으로 늘어난다.
따라서 주문 단위 분석과 상품 단위 분석을 구분해야 한다.
*/

SELECT
    o.order_id,
    o.customer_id,
    c.customer_unique_id,
    c.customer_state,
    c.customer_city,
    o.order_status,
    o.order_purchase_timestamp,
    o.purchase_month,
    o.delivery_days,
    o.delay_days,
    o.is_delayed,
    oi.order_item_id,
    oi.product_id,
    oi.seller_id,
    oi.price,
    oi.freight_value,
    COALESCE(t.product_category_name_english, p.product_category_name, 'unknown') AS product_category,
    s.seller_state,
    r.review_score
FROM orders_enriched o
LEFT JOIN customers c
    ON o.customer_id = c.customer_id
LEFT JOIN order_items oi
    ON o.order_id = oi.order_id
LEFT JOIN products p
    ON oi.product_id = p.product_id
LEFT JOIN category_translation t
    ON p.product_category_name = t.product_category_name
LEFT JOIN sellers s
    ON oi.seller_id = s.seller_id
LEFT JOIN order_reviews r
    ON o.order_id = r.order_id
WHERE o.order_status = 'delivered'
LIMIT 20;
"""

delivered_join_sample = pd.read_sql_query(query, conn)
delivered_join_sample

,order_id,customer_id,customer_unique_id,customer_state,customer_city,order_status,order_purchase_timestamp,purchase_month,delivery_days,delay_days,is_delayed,order_item_id,product_id,seller_id,price,freight_value,product_category,seller_state,review_score
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,7c396fd4830fd04220f754e42b4e5bff,SP,sao paulo,delivered,2017-10-02 10:56:33,2017-10,8.436574,-7.107488,0,1,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,29.99,8.72,housewares,SP,4.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,af07308b275d755c9edb36a90c618231,BA,barreiras,delivered,2018-07-24 20:41:37,2018-07,13.782037,-5.355729,0,1,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,118.70,22.76,perfumery,SP,4.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,3a653a41f6f9fc3d2a113cf8398680e8,GO,vianopolis,delivered,2018-08-08 08:38:49,2018-08,9.394213,-17.245498,0,1,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,159.90,19.22,auto,SP,5.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,7c142cf63193a1473d2e66489a9ae977,RN,sao goncalo do amarante,delivered,2017-11-18 19:28:06,2017-11,13.208750,-12.980069,0,1,d0b61bfb1de832b15ba9d266ca96e5b0,66922902710d126a0e7d26b0e3805106,45.00,27.20,pet_shop,MG,5.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,72632f0f9dd73dfee390c9b22eb56dd6,SP,santo andre,delivered,2018-02-13 21:18:39,2018-02,2.873877,-9.238171,0,1,65266b2da20d04dbe00c5c2d3bb7859e,2c9e548be18521d1c43cde1c582c6de8,19.90,8.72,stationery,SP,5.0
5,a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,80bb27c7c16e8f973207a5086ab329e2,PR,congonhinhas,delivered,2017-07-09 21:57:05,2017-07,16.542245,-5.543113,0,1,060cb19345d90064d1015407193c233d,8581055ce74af1daba164fdbd55a40de,147.90,27.36,auto,SP,4.0
6,6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,932afa1e708222e5821dac9cd5db4cae,RJ,nilopolis,delivered,2017-05-16 13:10:30,2017-05,9.989826,-11.461215,0,1,4520766ec412348b8d4caa5e8a18c464,16090f2ca825584b5a147ab24aa30c86,59.99,15.17,auto,SP,5.0
7,76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,39382392765b6dc74812866ee5ee92a7,RS,faxinalzinho,delivered,2017-01-23 18:29:09,2017-01,9.818762,-31.410995,0,1,ac1789e492dcd698c5c10b97a671243a,63b9ae557efed31d1f7687917d248a8d,19.90,16.05,furniture_decor,SP,1.0
8,e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,299905e3934e9e181bfb2e164dd4b4f8,SP,sorocaba,delivered,2017-07-29 11:55:02,2017-07,18.221852,-6.281597,0,1,9a78fb9862b10749a117f7fc3c31f051,7c67e1448b00f6e969d365cea6b010ab,149.99,19.77,office_furniture,SP,5.0
9,e6ce16cb79ec1d90b1da9085a6118aeb,494dded5b201313c64ed7f100595b95c,f2a85dec752b8517b5e58a06ff3cd937,RJ,rio de janeiro,delivered,2017-05-16 19:41:10,2017-05,12.650937,-8.528808,0,1,08574b074924071f4e201e151b152b4e,001cca7ae9ae17fb1caed9dfb1094831,99.00,30.53,garden_tools,ES,1.0


In [9]:
## 26. 분석 단위별 row 수 확인
query = """
/*
목적:
분석 단위별 데이터 규모를 확인한다.

- delivered_orders: 배송 완료 주문 수
- delivered_order_item_rows: 배송 완료 주문에 연결된 상품 row 수
- delivered_customers: 배송 완료 주문 고객 수
- delivered_unique_customers: 고유 고객 수
*/

SELECT
    COUNT(DISTINCT o.order_id) AS delivered_orders,
    COUNT(oi.order_id) AS delivered_order_item_rows,
    COUNT(DISTINCT o.customer_id) AS delivered_customer_ids,
    COUNT(DISTINCT c.customer_unique_id) AS delivered_unique_customers
FROM orders_enriched o
LEFT JOIN customers c
    ON o.customer_id = c.customer_id
LEFT JOIN order_items oi
    ON o.order_id = oi.order_id
WHERE o.order_status = 'delivered';
"""

analysis_unit_summary = pd.read_sql_query(query, conn)
analysis_unit_summary


## 27. 반복 구매 가능성 확인
query = """
/*
목적:
customer_unique_id 기준 반복 구매 규모를 확인한다.

해석 주의:
Olist 데이터는 반복 구매가 많지 않은 편으로 알려져 있다.
반복 구매 고객 비율이 낮다면 코호트/리텐션 분석을 메인 분석으로 두지 않는다.
*/

WITH customer_order_count AS (
    SELECT
        c.customer_unique_id,
        COUNT(DISTINCT o.order_id) AS order_count
    FROM orders o
    JOIN customers c
        ON o.customer_id = c.customer_id
    WHERE o.order_status = 'delivered'
    GROUP BY c.customer_unique_id
)

SELECT
    COUNT(*) AS unique_customers,
    SUM(CASE WHEN order_count >= 2 THEN 1 ELSE 0 END) AS repeat_customers,
    ROUND(
        100.0 * SUM(CASE WHEN order_count >= 2 THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS repeat_customer_rate_pct,
    AVG(order_count) AS avg_orders_per_customer,
    MAX(order_count) AS max_orders_per_customer
FROM customer_order_count;
"""

repeat_purchase_summary = pd.read_sql_query(query, conn)
repeat_purchase_summary

repeat_purchase_summary.to_csv(
    OUTPUT_DIR / "07_repeat_purchase_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

In [10]:
## 28. 1단계 검증 결과 요약 테이블 만들기
step1_summary = {
    "loaded_tables": len(dfs),
    "total_orders": len(dfs["orders"]),
    "delivered_orders": int(delivered_enriched.shape[0]),
    "delivered_order_rate_pct": round(
        delivered_enriched.shape[0] / len(dfs["orders"]) * 100,
        2
    ),
    "delayed_order_rate_pct": round(
        delivered_enriched["is_delayed"].mean() * 100,
        2
    ),
    "avg_delivery_days": round(
        delivered_enriched["delivery_days"].mean(),
        2
    ),
    "median_delivery_days": round(
        delivered_enriched["delivery_days"].median(),
        2
    ),
    "order_start_date": str(dfs["orders"]["order_purchase_timestamp"].min()),
    "order_end_date": str(dfs["orders"]["order_purchase_timestamp"].max()),
}

step1_summary_df = pd.DataFrame([step1_summary])
display(step1_summary_df)

step1_summary_df.to_csv(
    OUTPUT_DIR / "08_step1_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

## 29. DB 연결 종료
conn.close()
print("SQLite 연결 종료 완료")

,loaded_tables,total_orders,delivered_orders,delivered_order_rate_pct,delayed_order_rate_pct,avg_delivery_days,median_delivery_days,order_start_date,order_end_date
0,9,99441,96478,97.02,8.11,12.56,10.22,2016-09-04 21:15:19,2018-10-17 17:30:18


SQLite 연결 종료 완료
